## SETUPS ##

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


## SCRAPPING ##

In [2]:
ABBYSINIA_APP_ID = 'com.boa.boaMobileBanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    ABBYSINIA_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Abbysinia App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

Abbysinia App Info
App Title   : BoA Mobile
Current Score: 4.4006343
Total Ratings: 9,301
Total Reviews: 1,470
Installs     : 1,000,000+


In [3]:
# Step 2: Scrape reviews
print(f"Scraping reviews for Abbysinia...")

result, continuation_token = reviews(
    ABBYSINIA_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=400,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Abbysinia...
Collected 400 raw reviews


In [4]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: e1dfd8e3-02ad-4d83-80e8-0fa5ec2c06e3
  userName: Abebe Tesfa
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocLdN50Lw0hgBxJg3llmk4N7-8t-3KLCcXpX_1oh-BvvW3PtsA=mo
  content: Tilku
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 26.05.11
  at: 2026-05-18 03:16:59
  replyContent: None
  repliedAt: None
  appVersion: 26.05.11


In [5]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (400, 6)


,review_id,review,rating,date,bank,source
0,e1dfd8e3-02ad-4d83-80e8-0fa5ec2c06e3,Tilku,5,2026-05-18 03:16:59,Awash Bank,Google Play
1,ea419fbc-41fc-4211-b3b4-84e87d2952a4,the transaction is not working???? fix it,1,2026-05-18 01:13:03,Awash Bank,Google Play
2,0fe83567-9471-413c-b722-a24757bb5d82,sometimes The App Is not goibg through,3,2026-05-17 15:21:31,Awash Bank,Google Play
3,9dd3879e-2b0c-4058-8c8c-19f6b376f69c,"The worst app, also bank am begging for my own...",1,2026-05-16 12:35:22,Awash Bank,Google Play
4,5f69466d-ec06-4eb5-816c-296accffeff2,was Good 🙏,5,2026-05-16 00:10:06,Awash Bank,Google Play


In [6]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 400

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [7]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  232  ██████████████████████████████████████████████
  4 stars:   31  ██████
  3 stars:   14  ██
  2 stars:   13  ██
  1 stars:  110  ██████████████████████


In [8]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-18 03:16:59
1   2026-05-18 01:13:03
2   2026-05-17 15:21:31
3   2026-05-16 12:35:22
4   2026-05-16 00:10:06
5   2026-05-15 21:07:21
6   2026-05-15 15:01:09
7   2026-05-14 21:18:44
8   2026-05-12 11:50:32
9   2026-05-11 18:18:54

Date dtype: datetime64[us]
